In [2]:
%pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement anaconda-anon-usage (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for anaconda-anon-usage
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import json
from chunking import SectionAwareChunker, SemanticChunker
from embeddings import MiniLMEmbedding, QwenEmbedding
from vectore_store import FAISSVectorDB, ChromaVectorDB
from ranking_n_retrieval import Retriever
from llm_n_prompt import QwenLLM, PromptTemplate
from rag_pipeline import RAGPipeline

/Users/manish09/Desktop/ttm/CuisineRAG/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ==============================================================
# CONFIGURATION — change these to test different combinations
# ==============================================================
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

CHUNKER    = "section"  # "section" or "semantic"
EMBEDDING  = "minilm"   # "minilm"  or  "qwen"
VECTORDB   = "faiss"    # "faiss"   or  "chroma"
RETRIEVAL  = "combo2"   # "combo1"  or  "combo2"
DEVICE     = get_device()
FILEPATHS  = [
    "data/raw/south_asian_corpus.json",
    "data/raw/saved_wikibook_data.json",
    "data/raw/cuisines80.json"
]

In [5]:
def build_embedder(choice):
    if choice == "minilm":
        return MiniLMEmbedding(), 384
    elif choice == "qwen":
        return QwenEmbedding(), 768
    else:
        raise ValueError(f"Unknown embedder: {choice}. Choose 'minilm' or 'qwen'")


def build_vectordb(choice, dim):
    if choice == "faiss":
        return FAISSVectorDB(dim=dim)
    elif choice == "chroma":
        return ChromaVectorDB()
    else:
        raise ValueError(f"Unknown vectordb: {choice}. Choose 'faiss' or 'chroma'")


def build_retriever(choice, vectordb, documents):
    retriever = Retriever(vectordb, documents)
    retriever.active_combo = choice   # store choice on the object
    return retriever


In [6]:
def run_json_input_output(input_file_name, output_file_name):

    # print("\n" + "="*50)
    # print("       CuisineRAG — ChefBot")
    # print("="*50)
    # print(f"Chunker: {CHUNKER}")
    # print(f"  Embedding : {EMBEDDING}")
    # print(f"  VectorDB  : {VECTORDB}")
    # print(f"  Retrieval : {RETRIEVAL}")
    # print(f"  Device    : {DEVICE}")
    # print("="*50 + "\n")

    # --- build components based on config ---
    if CHUNKER == "semantic":
        chunker = SemanticChunker()       # reuses all-MiniLM-L6-v2
    else:
        chunker = SectionAwareChunker()
    embedder, dim = build_embedder(EMBEDDING)
    vectordb      = build_vectordb(VECTORDB, dim)
    prompt_builder = PromptTemplate()
    llm           = QwenLLM(device=DEVICE)

    # --- build pipeline (retriever added after indexing) ---

    pipeline = RAGPipeline(
        chunker        = chunker,
        embedder       = embedder,
        vectordb       = vectordb,
        retriever      = None,
        prompt_builder = prompt_builder,
        llm            = llm
    )

    # --- index the cuisine data ---

    import os

    INDEX_BIN  = "faiss_index.bin"
    DOCS_JSON  = "faiss_docs.json"

    if os.path.exists(INDEX_BIN) and os.path.exists(DOCS_JSON):
        # already indexed — just load from disk
        vectordb.load(INDEX_BIN, DOCS_JSON)
        pipeline.chunks = vectordb.documents
        print("Loaded existing index from disk.")
    else:
        # first run — index and save
        pipeline.index_data(FILEPATHS)
        vectordb.save(INDEX_BIN, DOCS_JSON)

    # --- build retriever with chunks for BM25 ---

    retriever = build_retriever(RETRIEVAL, vectordb, pipeline.chunks)
    pipeline.retriever = retriever

    with open(input_file_name) as input_file:
        query_list=json.load(input_file)['queries']

    print(f"\nProcessing {len(query_list)} queries from {input_file_name}...\n")

    results=[0]*len(query_list)
    for i,query in enumerate(query_list):
        query_id=query["query_id"]
        query_text=query["query"].strip()
        print(f"[{i+1}/{len(query_list)}] {query_text[:60]}...")
        answer,docs=pipeline.query(query_text)
        results[i]={
            "query_id":str(query_id),
            "query":str(query_text),
            "response":str(answer),
            "retrieved_context":[{"doc_id":chunk.metadata.get('doc_id', '?'),"text":chunk.page_content} for chunk in docs]
            }
        final=dict({"results": results})

    with open(output_file_name,'w') as output_json:
        json.dump(final, output_json, indent=2, ensure_ascii=False)

    print(f"\nDone! {len(results)} results saved to {output_file_name}")



Please put your input .json file in the **inputs_and_outputs** folder, this folder will also contain your output after you run the 
**run_json_input_output()** function.

In [ ]:
input_file_path = "inputs_and_outputs/input_payload_sample.json" #change as per your file name
output_file_path = "inputs_and_outputs/output_payload_sample.json" #change if needed or leave as is

: 

In [ ]:
if __name__ == "__main__":
    run_json_input_output(input_file_path, output_file_path)

Loading Qwen embedding model...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4374.14it/s]


Initializing FAISS...
Loading LLM...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 21708.10it/s]


Loaded 327 docs from data/raw/south_asian_corpus.json
Loaded 716 docs from data/raw/saved_wikibook_data.json
Loaded 6 docs from data/raw/cuisines80.json
Indexing 1,049 documents total...


### EVALUATION PIPELINE

In [14]:
%pip install pandas

  Using cached pandas-3.0.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp313-cp313-macosx_11_0_arm64.whl (9.9 MB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.0 MB/s  0:00:00

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import json
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_rag_pipeline(output_path, benchmark_path, output_csv='combined_evaluation_scores.csv'):
    """
    Evaluates a RAG pipeline's generation and retrieval performance.
    """
    # 1. Load the data
    with open(output_path, 'r') as f:
        output_data = json.load(f)

    with open(benchmark_path, 'r') as f:
        benchmark_data = json.load(f)

    # 2. Create mappings from the benchmark dataset
    benchmark_refs = {item['query']: item['reference'] for item in benchmark_data}
    benchmark_docs = {item['query']: set(item['relevant_doc_ids']) for item in benchmark_data}

    evaluation_results = []
    smoother = SmoothingFunction().method1

    # 3. Iterate through your RAG system's outputs once
    for item in output_data['results']:
        query = item['query']
        response = item['response']
        retrieved_docs = [doc['doc_id'] for doc in item['retrieved_context']]
        
        # Ensure the query exists in our benchmark dataset before evaluating
        if query in benchmark_refs and query in benchmark_docs:
            reference = benchmark_refs[query]
            true_docs = benchmark_docs[query]
            
            # ==========================================
            #           GENERATION METRICS
            # ==========================================
            
            # BLEU Score Calculation
            ref_tokens = reference.lower().split()
            resp_tokens = response.lower().split()
            bleu = sentence_bleu([ref_tokens], resp_tokens, smoothing_function=smoother)
            
            # Simple ROUGE-1 (Unigram) Calculation
            ref_set = set(ref_tokens)
            resp_set = set(resp_tokens)
            overlap = ref_set.intersection(resp_set)
            
            rouge_1_recall = len(overlap) / len(ref_set) if len(ref_set) > 0 else 0
            rouge_1_precision = len(overlap) / len(resp_set) if len(resp_set) > 0 else 0
            
            if rouge_1_precision + rouge_1_recall > 0:
                rouge_1_f1 = 2 * (rouge_1_precision * rouge_1_recall) / (rouge_1_precision + rouge_1_recall)
            else:
                rouge_1_f1 = 0

            # ==========================================
            #           RETRIEVAL METRICS
            # ==========================================
            
            # Intersection of retrieved vs true documents
            hits = set(retrieved_docs).intersection(true_docs)
            
            # Retrieval Precision & Recall
            precision = len(hits) / len(retrieved_docs) if retrieved_docs else 0
            recall = len(hits) / len(true_docs) if true_docs else 0
            
            # Mean Reciprocal Rank (MRR)
            mrr = 0
            for rank, doc_id in enumerate(retrieved_docs, start=1):
                if doc_id in true_docs:
                    mrr = 1.0 / rank
                    break
                    
            # Store all metrics for this specific query
            evaluation_results.append({
                'query': query,
                'bleu_score': bleu,
                'rouge1_recall': rouge_1_recall,
                'rouge1_precision': rouge_1_precision,
                'rouge1_f1': rouge_1_f1,
                'precision': precision,
                'recall': recall,
                'mrr': mrr
            })

    # 4. Save to a single Dataframe & CSV
    df = pd.DataFrame(evaluation_results)
    df.to_csv(output_csv, index=False)

    # 5. Output the Final Results
    print("--- GENERATION METRICS ---")
    print(f"Mean BLEU Score: {df['bleu_score'].mean():.4f}")
    print(f"Mean ROUGE-1 F1: {df['rouge1_f1'].mean():.4f}")
    print(f"Mean ROUGE-1 Recall: {df['rouge1_recall'].mean():.4f}\n")

    print("--- RETRIEVAL METRICS ---")
    print(f"Mean Precision: {df['precision'].mean():.4f}")
    print(f"Mean Recall: {df['recall'].mean():.4f}")
    print(f"Mean MRR: {df['mrr'].mean():.4f}")

    return df

# Run the pipeline

In [18]:
# Replace these filenames with the actual paths to your JSON files
if __name__ == "__main__":
    evaluate_rag_pipeline(
        output_path='inputs_and_outputs/output_payload_sample.json', 
        benchmark_path='data/benchmark_dataset.json'
    )

--- GENERATION METRICS ---
Mean BLEU Score: 0.0914
Mean ROUGE-1 F1: 0.3229
Mean ROUGE-1 Recall: 0.5669

--- RETRIEVAL METRICS ---
Mean Precision: 0.0000
Mean Recall: 0.0000
Mean MRR: 0.0000
